# EDS � Full-sequence stride results
Run all cells to view saved scores. Each dataset mean weights the available EDS test sequences equally. Stride 1 uses all frames, and strides 2-4 retain approximately one half, one third and one quarter. Set REPORT_ID to select a run. The chart also works as a standalone offline HTML file.

In [ ]:
from pathlib import Path
import html
import json
import numpy as np
import pandas as pd
from IPython.display import HTML, FileLink, display

REPORT_ID = "8fa73a5e847f7f56"  # Set to None to use the most recently written CSV.
REPORT_ROOT = None  # Optional path to reports/experiments/eds/test
METRICS = None  # None shows every metric saved in this report.
SEQUENCES = ["02_rocket_earth_light", "06_ziggy_and_fuzz", "10_office", "14_ziggy_in_the_arena"]
STEPS = [1, 2, 3, 4]

VIEW_MS = [0, 1, 2, 3]
RATE_SEQUENCE = "10_office"  # Choose an EDS test sequence for the time series.
RATE_BIN_MS = 1.0
DATA_ROOT = None  # Optional dataset root containing eds/test.


In [ ]:
if REPORT_ROOT is None:
    relative = Path("reports/experiments/eds/test")
    roots = [p / relative for p in (Path.cwd(), *Path.cwd().parents) if (p / relative).is_dir()]
    if not roots:
        raise FileNotFoundError("Open this notebook inside the benchmark repo or set REPORT_ROOT.")
    root = roots[0]
else:
    root = Path(REPORT_ROOT).expanduser()

files = sorted(root.glob("full_sequence_strides*/*/per_sequence.csv"),
               key=lambda p: p.stat().st_mtime_ns, reverse=True)
matches = [p for p in files if REPORT_ID is None or p.parent.name == REPORT_ID]
if not matches:
    raise FileNotFoundError(f"No matching report under {root}. Available: {[str(p.relative_to(root)) for p in files]}")
path = matches[0]
print("Reading", path)
df = pd.read_csv(path)
keys = ["metric", "method", "stride", "sequence"]
df = df[df.sequence.isin(SEQUENCES) & df.stride.isin(STEPS)].copy()
metrics = list(dict.fromkeys(df.metric)) if METRICS is None else list(METRICS)
if not metrics or set(metrics) - set(df.metric):
    raise ValueError(f"Choose from the saved metrics: {sorted(df.metric.unique())}")
df = df[df.metric.isin(metrics)].copy()
if df.duplicated(keys).any():
    raise ValueError("Duplicate scores for a metric, method, stride and sequence.")
df["value"] = pd.to_numeric(df.value, errors="coerce")
accepted = df.status.eq("ok") | (df.metric.eq("fano_factor") & df.status.eq("partial_support"))
df["valid"] = df.value.where(accepted & np.isfinite(df.value))
agg = df.groupby(["metric", "method", "stride"], as_index=False).agg(
    mean=("valid", "mean"), n=("valid", "count"))
methods = sorted(df.method.unique(), key=lambda m: (m != "ground_truth", m))
display(pd.crosstab(df.metric, df.status))

## Dataset means by stride
Fano error uses jointly active cells. Its mean includes finite partial_support scores, so methods can have different spatial-temporal coverage. Each cell shows the mean and the number of contributing sequences. A dash indicates no valid score. The three best distinct scores are shaded for error metrics and event count ratio (closest to 1), among models with complete coverage. Descriptive metrics and recorded GT are shown without ranking.

In [ ]:
lower = {"fano_factor", "chamfer", "chamfer_full", "chamfer_penalty", "chamfer_count", "iei_distance"}
for step in STEPS:
    group = agg[agg.stride.eq(step)]
    values = group.pivot(index="method", columns="metric", values="mean").reindex(index=methods, columns=metrics)
    counts = group.pivot(index="method", columns="metric", values="n").reindex_like(values).fillna(0)
    colors = {}
    for metric in metrics:
        if metric not in lower and not metric.startswith(("voxel_", "pmse_")) and metric != "event_count_ratio":
            continue
        eligible = values.loc[(counts[metric] == len(SEQUENCES)) & (values.index != "ground_truth"), metric].dropna()
        errors = (eligible - 1).abs() if metric == "event_count_ratio" else eligible
        for rank, val in enumerate(sorted(errors.unique())[:3]):
            for method in errors.index[errors == val]:
                colors[method, metric] = ["#9dc9ff", "#c4dfff", "#e2efff"][rank]
    rows = []
    for method in methods:
        cells = []
        for metric in metrics:
            value = values.at[method, metric]
            label = "-" if pd.isna(value) else f"{value:.5g} ({int(counts.at[method, metric])}/{len(SEQUENCES)})"
            color = colors.get((method, metric))
            style = f' style="background:{color};color:#102641"' if color else ""
            cells.append(f"<td{style}>{label}</td>")
        rows.append("<tr><th>" + html.escape(method) + "</th>" + "".join(cells) + "</tr>")
    headers = "".join(f"<th>{html.escape(metric)}</th>" for metric in metrics)
    display(HTML(f"<h3>Stride {step} � M={step-1}</h3><div style='overflow-x:auto'><table>"
                 f"<thead><tr><th>Method</th>{headers}</tr></thead><tbody>{''.join(rows)}</tbody></table></div>"))

## Interactive comparison
Choose a metric and either the dataset mean or one sequence. Hover over points for exact scores and coverage. Missing scores leave gaps. The chart includes recorded GT wherever the report supplies it.

In [ ]:
import plotly.graph_objects as go

# Embed each plot with its own controls. One local copy of Plotly serves the entire dashboard.
from plotly.offline import get_plotlyjs

panels = []
options = []
for i, metric in enumerate(metrics):
    fig = go.Figure()
    choices = ["Dataset mean", *SEQUENCES]
    for choice in choices:
        for method in methods:
            if choice == "Dataset mean":
                subset = agg[(agg.metric == metric) & (agg.method == method)].set_index("stride").reindex(STEPS)
                y = subset["mean"]
                info = [f"{int(n)}/{len(SEQUENCES)} sequences" for n in subset.n.fillna(0)]
            else:
                subset = df[(df.metric == metric) & (df.method == method) & (df.sequence == choice)].set_index("stride").reindex(STEPS)
                y = subset.valid
                info = subset.status.fillna("no saved row").tolist()
            fig.add_scatter(x=STEPS, y=y, mode="lines+markers", connectgaps=False,
                            name="Recorded GT" if method == "ground_truth" else method,
                            visible=choice == choices[0], customdata=info,
                            line=dict(dash="dot" if method == "ground_truth" else "solid"),
                            hovertemplate="Stride %{x} � %{y:.6g}<br>%{customdata}<extra>%{fullData.name}</extra>")
    buttons = [dict(label=c, method="update", args=[dict(visible=[
        j // len(methods) == k for j in range(len(fig.data))])]) for k, c in enumerate(choices)]
    fig.update_layout(title=metric, template="plotly_white", height=510, margin=dict(t=115),
                      xaxis=dict(title="Input frame stride", tickvals=STEPS),
                      yaxis_title=metric, updatemenus=[dict(buttons=buttons, x=0, y=1.20)])
    if metric == "event_count_ratio":
        fig.add_hline(y=1, line_dash="dot")
    options.append(f'<option value="{i}">{html.escape(metric)}</option>')
    plot = fig.to_html(full_html=False, include_plotlyjs=False, config={"responsive": True})
    panels.append(f'<section id="p{i}" {"hidden" if i else ""}>{plot}</section>')

dashboard = ("<!doctype html><meta charset=utf-8><style>body{font:16px system-ui;color:#173354;background:white}"
             "select{font:inherit}section[hidden]{display:none}</style><script>" + get_plotlyjs() + "</script>"
             + "<label>Metric <select id='metric'>" + "".join(options) + "</select></label>" + "".join(panels)
             + """<script>document.getElementById('metric').onchange=function(){
                 document.querySelectorAll('section').forEach(s=>s.hidden=s.id!=='p'+this.value);
                 Plotly.Plots.resize(document.querySelector('section:not([hidden]) .js-plotly-plot'));
             };</script>""")
out_dir = Path.cwd() / "stride_figures" / path.parent.parent.name / path.parent.name
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / "stride_comparison.html"
out.write_text(dashboard, encoding="utf-8")
display(FileLink(str(out.relative_to(Path.cwd()))))
display(HTML('<iframe title="EDS stride comparison" width="100%" height="600" style="border:0" srcdoc="'
             + html.escape(dashboard, quote=True) + '"></iframe>'))

## ISI distribution by M
Each curve is the equally weighted mean of the saved per-sequence empirical CDFs. Hover for the fraction of intervals at or below each ISI and the number of contributing sequences. The grid includes zero for simultaneous events.
The combined figure below the individual plots has independent M and method checkboxes. Color identifies the method and line style identifies M. The shared GT appears once.

In [ ]:
source_rows = pd.read_csv(path)
source_rows = source_rows[source_rows.sequence.isin(SEQUENCES)]
steps = [m + 1 for m in VIEW_MS]
from plotly.colors import qualitative
palette = {name: qualitative.Plotly[i % len(qualitative.Plotly)] for i, name in enumerate(methods)}
def line_style(method):
    if method == "ground_truth":
        return dict(color="#161616", dash="dash", width=2.5)
    return dict(color=palette.get(method), width=1.5)

def show_temporal(fig, stem, controls=None):
    target = out_dir / (stem + ".html")
    markup = fig.to_html(include_plotlyjs=True, post_script=controls, config={"responsive": True})
    target.write_text(markup, encoding="utf-8")
    display(FileLink(str(target.relative_to(Path.cwd()))))
    display(HTML('<iframe width="100%" height="660" title="'+html.escape(stem)+'" srcdoc="'
                 + html.escape(markup, quote=True) + '"></iframe>'))

# Each trace carries its method and M for the offline HTML checkboxes.
overlay_controls = """const plot = document.getElementById('{plot_id}');
const controls = document.createElement('div');
plot.before(controls);
const tags = plot.data.map(t => t.meta);
function checkboxes(title, values) {
    const group = document.createElement('fieldset');
    const legend = document.createElement('legend');
    legend.textContent = title;
    group.append(legend);
    controls.append(group);
    const boxes = new Map();
    for (const value of values) {
        const label = document.createElement('label');
        const box = document.createElement('input');
        box.type = 'checkbox';
        box.checked = true;
        box.addEventListener('change', update);
        label.append(box, document.createTextNode(String(value)));
        label.style.marginRight = '1em';
        group.append(label);
        boxes.set(value, box);
    }
    return boxes;
}
const ms = checkboxes('M', [...new Set(tags.filter(t => t.m !== null).map(t => t.m))].sort((a,b) => a-b));
const models = checkboxes('Methods', [...new Set(tags.map(t => t.method))]);
function update() {
    const visible = tags.map(t => models.get(t.method).checked && (t.m === null || ms.get(t.m).checked));
    Plotly.restyle(plot, {visible});
}"""
dash_by_m = {0: "dot", 1: "solid", 2: "dash", 3: "dashdot"}

def add_to_overlay(overlay, figure, m):
    for trace in figure.data:
        method = trace.name.split(" (")[0]
        gt = method == "ground_truth"
        if gt and any(t.meta["m"] is None for t in overlay.data):
            continue
        overlay.add_trace(trace)
        added = overlay.data[-1]
        added.update(name=trace.name if gt else f"{trace.name} · M={m}",
                     meta=dict(method=method, m=None if gt else m))
        added.line.dash = "longdash" if gt else dash_by_m[m]
    overlay.update_layout(figure.layout)
    overlay.update_layout(legend=dict(itemclick=False, itemdoubleclick=False))

isi = source_rows[source_rows.metric.eq("isi_distribution") & source_rows.status.eq("ok")]
if isi.empty or "array_path" not in isi:
    print("No saved ISI distribution arrays listed for this selection.")
else:
    upper = max(float(isi.duration_s.max()) * 1000, 1e-6)
    grid = np.r_[0, np.geomspace(1e-6, upper, 200)]
    # A log-like scale with a finite origin keeps simultaneous events visible.
    def axis_position(ms):
        return np.log10(1 + np.asarray(ms) / 0.001)
    tick_ms = [v for v in (0, .001, .01, .1, 1, 10, 100, 1000, 10000, 100000) if v <= upper]
    overlay = go.Figure()
    for step in steps:
        fig = go.Figure()
        for method, group in isi[isi.stride.eq(step)].groupby("method"):
            curves = []
            for row in group.itertuples():
                array = path.parent.parent / str(row.array_path)
                if not array.is_file():
                    print("Missing ISI array:", array)
                    continue
                with np.load(array, allow_pickle=False) as archive:
                    intervals = archive["isi_ms"].ravel()
                if intervals.size:
                    intervals.sort()
                    curves.append(np.searchsorted(intervals, grid, side="right") / intervals.size)
            if curves:
                fig.add_scatter(x=axis_position(grid), customdata=grid, y=np.mean(curves, axis=0), mode="lines", line=line_style(method),
                                name=f"{method} ({len(curves)}/{len(SEQUENCES)})",
                                hovertemplate="%{customdata:.5g} ms · fraction %{y:.4f}<extra>%{fullData.name}</extra>")
        if fig.data:
            fig.update_layout(title=f"ISI cumulative distribution · M={step-1}", template="plotly_white",
                              xaxis_title="ISI (ms)", yaxis_title="Mean cumulative fraction",
                              xaxis=dict(tickvals=axis_position(tick_ms), ticktext=[str(v) for v in tick_ms]),
                              yaxis_range=[0, 1.02], height=475)
            add_to_overlay(overlay, fig, step - 1)
            show_temporal(fig, f"isi_m{step-1}")
        else:
            print(f"No usable ISI arrays for M={step-1}.")
    if overlay.data:
        overlay.update_layout(title="ISI cumulative distribution across M")
        show_temporal(overlay, "isi_compare_m", overlay_controls)


## Event rate over time by M
Select RATE_SEQUENCE and RATE_BIN_MS above. Each plot compares recorded GT with saved predictions on the scoring interval from this report. Rates are events per second. V2V supplies signed voxels without individual timestamps.
The combined figure below the individual plots has independent M and method checkboxes. Color identifies the method and line style identifies M. The shared GT appears once.

In [ ]:
import sys
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "experiments/score_windows.py").is_file())
sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "experiments"))
from score_windows import Window, bench
from benchmarks.event_rate_views_v2 import rate_arrays

data_root = Path(DATA_ROOT or bench.v2.old.setting("EVTWEEN_DATA", bench.v2.old.LAB / "evtween/data"))
source = data_root / "eds/test" / RATE_SEQUENCE
selected = source_rows[source_rows.sequence.eq(RATE_SEQUENCE) & source_rows.stride.isin(steps)]
bounds = selected[["last_scored_frame", "duration_s"]].drop_duplicates()
if len(bounds) != 1:
    raise ValueError(f"Expected one shared scoring interval for {RATE_SEQUENCE}; found {len(bounds)}.")
files, ts, _, base, _ = bench.v2.sequence_info(source)
last = int(bounds.iloc[0].last_scored_frame)
window = Window((source, files, ts, base), np.arange(last + 1), {})
stop = float(window.geometry[0][-1])
if not np.isclose(stop, bounds.iloc[0].duration_s, rtol=0, atol=1e-9):
    raise ValueError("Ground truth duration differs from the report.")
gt = rate_arrays(window.gt, 0., stop, RATE_BIN_MS)
del window
overlay = go.Figure()
for step in steps:
    subset = selected[selected.stride.eq(step)]
    if subset.empty:
        print(f"No report rows for M={step-1}.")
        continue
    indices = np.arange(0, len(ts), step)
    fig = go.Figure()
    for method in ["ground_truth", *sorted(set(subset.method) - {"ground_truth", "v2v"})]:
        if method == "ground_truth":
            rate = gt
        else:
            if step == 1:
                folder = repo / "data_test_evt/eds/test" / method / RATE_SEQUENCE
            else:
                m = step - 1
                folder = (repo / "data_test_exp/eds" / method / RATE_SEQUENCE / f"stride_m{m}"
                          / f"k_{len(indices)}" / f"m_{m}" / f"frames_0_{indices[-1]}")
            if not all((folder / f).is_file() for f in ("input.json", "events.npz")):
                print(f"M={step-1}: missing saved stream or metadata for {method}")
                continue
            meta = json.loads((folder / "input.json").read_text())
            if (meta["indices"] != indices.tolist() or meta["crop"] != base["crop"]
                or tuple(Path(meta["source"]).parts[-3:]) != ("eds", "test", RATE_SEQUENCE)
                or meta["absolute_t0"] != ts[0] or meta["duration"] != ts[indices[-1]] - ts[0]):
                print(f"M={step-1}: incompatible prediction metadata for {method}")
                continue
            events = bench.read_events(folder / "events.npz", meta)
            rate = rate_arrays(events, 0., stop, RATE_BIN_MS)
            del events
        fig.add_scatter(x=rate["edges"], y=np.r_[rate["total"], rate["total"][-1]], mode="lines",
                        line=dict(**line_style(method), shape="hv"), name=method,
                        hovertemplate="%{x:.5f} s · %{y:,.0f} events/s<extra>%{fullData.name}</extra>")
    fig.update_layout(title=f"{RATE_SEQUENCE} · M={step-1} · {RATE_BIN_MS:g} ms bins",
                      template="plotly_white", height=475, xaxis_title="Time (s)", yaxis_title="Events/s")
    add_to_overlay(overlay, fig, step - 1)
    show_temporal(fig, f"rate_{RATE_SEQUENCE}_m{step-1}")
if overlay.data:
    overlay.update_layout(title=f"{RATE_SEQUENCE} · event rate across M · {RATE_BIN_MS:g} ms bins")
    show_temporal(overlay, f"rate_{RATE_SEQUENCE}_compare_m", overlay_controls)
